In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm
from pathlib import Path
import warnings
import copy
import csv
import random

from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

np.random.seed(42)
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Paths
ROOT = Path("Amazon_products")
TRAIN_CORPUS_PATH = ROOT / "train" / "train_corpus.txt"
TEST_CORPUS_PATH  = ROOT / "test" / "test_corpus.txt"
CLASS_PATH        = ROOT / "classes.txt"

EMB_DIR      = Path("SilverGeneration/Embeddings")
X_ALL_PATH   = EMB_DIR / "X_train_test_mini.pt" # Train + Test embeddings
LABEL_EMB_PATH = EMB_DIR / "labels_mini.pt"

MODEL_SAVE = Path("Models")
MODEL_SAVE.mkdir(exist_ok=True)
MODEL_PATH = MODEL_SAVE / "classifierSelft.pt"
CLASS_HIERARCHY_PATH = ROOT / "class_hierarchy.txt"

label_emb = torch.load(LABEL_EMB_PATH).float().to(device)
print("Label embeddings:", label_emb.shape)

# Load useful data (like silver gen)
def load_classic(path):
    id2text = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            pid, text = line.strip().split("\t", 1)
            id2text[int(pid)] = text
    return id2text

def load_multilabel(path):
    """Load multi-label data into {id: [labels]} dictionary -> for class_hierarchy"""
    id2labels = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                pid, label = parts
                pid = int(pid)
                label = int(label)
                if pid not in id2labels:
                    id2labels[pid] = []
                id2labels[pid].append(label)
    return id2labels

id2text_train = load_classic(TRAIN_CORPUS_PATH)
id2text_test  = load_classic(TEST_CORPUS_PATH)
train_ids = list(id2text_train.keys())
test_ids  = list(id2text_test.keys())
n_train = len(train_ids)
n_test  = len(test_ids)
print(f"Train IDs: {n_train} | Test IDs: {n_test}")

# Load X_all + split into X_train & X_test
data = torch.load(X_ALL_PATH, weights_only=False)

# ensure tensor (check)
if isinstance(data, np.ndarray):
    data = torch.from_numpy(data)
elif isinstance(data, list):
    data = torch.stack(data)

X_all = data.float().to(device)
X_train = X_all[:n_train]
X_test  = X_all[n_train:]
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

# Load Class names
classes = {}
with open(CLASS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        cid, cname = line.strip().split("\t")
        classes[int(cid)] = cname

n_classes = len(classes) # 531
print(n_classes)

pid2idx = {pid: i for i, pid in enumerate(train_ids)}

class2hierarchy = load_multilabel(CLASS_HIERARCHY_PATH)
print(class2hierarchy)



In [ ]:
# Load Silver labels
with open("SilverGeneration/SilverCombo/silver_train_mpnet.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

silver_labels = {int(pid): data["labels"] for pid, data in raw.items()}

# Compute similarities
X_train_norm = F.normalize(X_train, p=2, dim=1).to(device)
label_emb_norm = F.normalize(label_emb, p=2, dim=1).to(device)
all_similarities = torch.matmul(X_train_norm, label_emb_norm.T).cpu()

# Map scores to silver labels
silver_with_scores = {}
for idx, pid in enumerate(train_ids):
    if pid not in silver_labels:
        continue

    labels = silver_labels[pid]
    sims = all_similarities[idx]

    label_scores = [float(sims[label]) for label in labels]

    silver_with_scores[pid] = {
        "labels": labels,
        "scores": label_scores,
        "max_score": max(label_scores),
    }

# Extract quartiles
max_scores = [info["max_score"] for info in silver_with_scores.values()]

Q1 = np.quantile(max_scores, 0.25)
Q2 = np.quantile(max_scores, 0.50)
Q3 = np.quantile(max_scores, 0.75)

print(f"Q1(25%): {Q1:.4f}")
print(f"Q2(50%): {Q2:.4f}")
print(f"Q3(75%): {Q3:.4f}")

# Choose threshold
threshold = Q2  # Top 25% more confident
print(f"\nUsing threshold: {threshold:.4f}")

# Split pseudo-labeled vs unlabeled
pseudo_pids = [pid for pid, info in silver_with_scores.items() if info["max_score"] >= threshold]

unlabeled_pids = [pid for pid, info in silver_with_scores.items() if info["max_score"] < threshold]

print(f"\nPseudo-labeled count : {len(pseudo_pids)}")
print(f"Unlabeled count : {len(unlabeled_pids)}")

# Build final datasets
silver_dataset = {pid: silver_with_scores[pid]["labels"] for pid in pseudo_pids}

unlabeled_dataset = unlabeled_pids

In [ ]:
class MultiLabelDataset(Dataset):
    """
    Simple PyTorch dataset for multi-label classification.
    Takes a list of product IDs and a dict pid -> list of labels,
    and returns (embedding, multi-hot label vector) for each item.
    """
    def __init__(self, pids, labels_dict):
        self.pids = pids
        self.labels = labels_dict

    def __len__(self):
        return len(self.pids)

    def __getitem__(self, idx):
        pid = self.pids[idx]
        emb = X_train[pid2idx[pid]]

        y = torch.zeros(n_classes)
        for c in self.labels[pid]:
            if 0 <= c < n_classes:
                y[c] = 1.0 # one-hot multi-label vector

        return {"X": emb, "y": y}
    

class UnlabeledEmbeddingDataset(Dataset):
    """
    Dataset for unlabeled samples used in self-training.
    """
    def __init__(self, pids):
        self.pids = pids  # list of unlabeled product IDs

    def __len__(self):
        return len(self.pids)

    def __getitem__(self, idx):
        pid = self.pids[idx]
        emb = X_train[pid2idx[pid]]

        return {"X": emb, "pid": pid}    

# TRAIN / VAL splits
train_p, val_p = train_test_split(list(silver_dataset.keys()),test_size=0.15,random_state=42)

train_dataset = MultiLabelDataset(train_p, silver_dataset)
val_dataset   = MultiLabelDataset(val_p,   silver_dataset)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)

print(len(train_dataset))


In [ ]:
class InnerProductClassifier(nn.Module):
    """
    A simple classifier that projects input embeddings and
    scores each label via inner product with its embedding.
    """
    def __init__(self, input_dim, label_embeddings, dropout=0.2, trainable_label_emb=False):
        super().__init__()

        D = label_embeddings.size(1)

        self.proj = nn.Linear(input_dim, D)
        self.dropout = nn.Dropout(dropout)

        # Label embeddings can be fixed or trainable (for innerproduct we'll prefer fixed cause 
        # it preserves semantic meaning from pretrained embeddings and reduces overfitting risk 
        # on noisy silver labels)        
        if trainable_label_emb:
            self.label_emb = nn.Parameter(label_embeddings.clone())
        else:
            self.register_buffer("label_emb", label_embeddings.clone())

    def forward(self, x, use_dropout=True):
        # project input
        x_proj = self.proj(x)
        
        if use_dropout:
            x_proj = self.dropout(x_proj)
        
        # normalize
        x_proj = F.normalize(x_proj, dim=1)
        label_emb = F.normalize(self.label_emb, dim=1)
        
        # Scale to expand sigmoid range: normalized inner products ∈ [-1,1] -> need scaling for confident predictions
        logits = (x_proj @ label_emb.T) * 30.0
        return logits


In [ ]:
# Evaluation metrics -> see paper given
def precision_at_1(y_true, scores):
    """
    Computes Precision@1 for multi-label classification.
    For each sample, we check if the top-1 predicted class is actually a true label.
    """
    correct = 0
    for yt, sc in zip(y_true, scores):
        top1 = sc.argmax()
        if yt[top1] == 1:
            correct += 1
    return correct / len(y_true)


def precision_at_3(y_true, scores):
    """
    Computes Precision@3 for multi-label classification.
    For each sample, we check how many of the top-3 predicted classes
    match the true labels, and average over all samples.
    """
    total_correct = 0
    for yt, sc in zip(y_true, scores):
        top3 = sc.argsort()[-3:][::-1]   # indices of best 3 scores
        total_correct += yt[top3].sum()
    return total_correct / (3 * len(y_true))


# MLPs generate high-magnitude logits (leading to sigmoid outputs near 0 or 1), so thresholds around 0.4–0.6 perform best.
# In contrast, InnerProduct and GNN models use cosine-like similarity scores,which are naturally compressed in the range 0.15–0.40 after sigmoid, thus requiring lower thresholds (≈0.20–0.30).
def evaluate(model, loader, thr=0.15):
    """
    Evaluate a multi-label classifier on a dataloader.
    Applies sigmoid to get probabilities, turns them into binary labels
    using a threshold, and computes sample-wise and macro F1 scores.
    """
    model.eval()
    preds, labels = [], []
    all_scores = []   # predicted scores for each example
    all_true = []     # true binary vectors

    with torch.no_grad():
        for batch in loader:
            X = batch["X"]
            y = batch["y"].numpy()

            logits = model(X)
            prob = torch.sigmoid(logits).cpu().numpy()

            # For threshold metrics
            pred = (prob > thr).astype(int)
            preds.extend(pred)
            labels.extend(y)

            # For ranking metrics
            all_scores.extend(prob)
            all_true.extend(y)

    # F1 metrics
    f1s = f1_score(labels, preds, average="samples")
    f1m = f1_score(labels, preds, average="macro")

    # Ranking metrics
    P1 = precision_at_1(all_true, all_scores)
    P3 = precision_at_3(all_true, all_scores)

    return f1s, f1m, P1, P3


In [ ]:
# Classic training without techniques of regularization only early stopping and bestf1 improvement
model = InnerProductClassifier(X_train.size(1), label_emb, 0.2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

epochs = 100
val_f1_list = []
val_p1_list = []
val_p3_list = []
best_f1 = 0
best_state = None
patience = 5
wait = 0

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0

    # Training
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):
        X = batch["X"].to(device)
        y = batch["y"].to(device)

        logits = model(X)
        # We use BCE-with-logits because we are using a multi-label system: each class must have its own probability. 
        # This loss automatically applies a sigmoid to each class and calculates the bit error for each one.
        loss = F.binary_cross_entropy_with_logits(logits, y) 

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    f1s, f1m, p1, p3 = evaluate(model, val_loader)
    val_f1_list.append(f1s)
    val_p1_list.append(p1)
    val_p3_list.append(p3)

    print(f"[Epoch {epoch}] loss={avg_loss:.4f} | F1={f1s:.4f}")

    # Save best
    if f1s > best_f1: # we compare on f1 sample like kaggle
        best_f1 = f1s
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, MODEL_PATH)
        print(f"New best model (F1={best_f1:.4f})")
        wait = 0
    else:
        wait += 1 # no improvement this epoch
        print(f" No improvement for {wait} epoch(s).")
    
    if wait >= patience:
        print(f"\nEarly stopping triggered after {epoch} epochs.")
        break

print("\nTraining finished")
print(f"Best validation F1 = {best_f1:.4f}")

# Load best model
model.load_state_dict(best_state)

save_model = copy.deepcopy(model)


In [ ]:
def consistency_loss(log_s, log_t):
    """MSE between teacher/student probabilities"""
    ps = torch.sigmoid(log_s)
    pt = torch.sigmoid(log_t)
    return F.mse_loss(ps, pt)

def bce_with_logits_smooth(logits, targets, smoothing=0.05):
    """
    BCE with logits + label smoothing.
    Helps reduce overconfident predictions and improves stability in multi-label tasks.
    """
    targets = targets * (1 - smoothing) + 0.5 * smoothing
    return F.binary_cross_entropy_with_logits(logits, targets)

# See former assignments for this
def ema_update(teacher, student, alpha):
    """teacher = alpha*teacher + (1-alpha)*student"""
    for p_t, p_s in zip(teacher.parameters(), student.parameters()):
        p_t.data.mul_(alpha).add_(p_s.data, alpha=1 - alpha)


In [ ]:
# --- WARMUP / PRETRAINING FIXÉ AVEC EMA ---

warmup_epochs = 10
val_f1_list2 = []
val_p1_list2 = []
val_p3_list2 = []

# Student = modèle principal
model = InnerProductClassifier(
    X_train.size(1),
    label_emb,
    dropout=0.4    # IMPORTANT : même dropout qu'en self-training
).to(device)

# Teacher = EMA du student
teacher = InnerProductClassifier(
    X_train.size(1),
    label_emb,
    dropout=0.0     # jamais de dropout dans le teacher
).to(device)

# Initialisation du teacher = student
teacher.load_state_dict(model.state_dict())

optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
alpha_ema = 0.995  # EMA plus lent = teacher plus stable

best_f1 = 0
best_state = None
wait = 0
patience = 10

print("\n===== WARM-UP WITH EMA START =====\n")

for epoch in range(1, warmup_epochs + 1):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Warmup Epoch {epoch}/{warmup_epochs}"):
        X = batch["X"].to(device)
        y = batch["y"].to(device)

        # ---- 1) Forward student ----
        logits = model(X, use_dropout=True)
        loss = F.binary_cross_entropy_with_logits(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # ---- 2) EMA UPDATE OF TEACHER ----
        with torch.no_grad():
            for p_t, p_s in zip(teacher.parameters(), model.parameters()):
                p_t.data.mul_(alpha_ema).add_(p_s.data, alpha=1 - alpha_ema)

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # ---- Validation ----
    model.eval()
    f1s, f1m, p1, p3 = evaluate(model, val_loader)
    val_p1_list2.append(p1)
    val_p3_list2.append(p3)
    val_f1_list2.append(f1s)

    print(f"[Warmup Epoch {epoch}] loss={avg_loss:.4f} | F1={f1s:.4f}")

    # Save best student
    if f1s > best_f1:
        best_f1 = f1s
        best_state = copy.deepcopy(model.state_dict())
        print(f"New best student (F1={best_f1:.4f})")
        wait = 0
    else:
        wait += 1
        print(f"No improvement for {wait} epoch(s).")

    if wait >= patience:
        print("\nEarly stopping warm-up.")
        break

print("\n===== END OF WARMUP =====")
print(f"Best warm-up F1 = {best_f1:.4f}")

# Charge les meilleurs poids du student
model.load_state_dict(best_state)

# IMPORTANT : le teacher doit aussi partir du best student stable
teacher.load_state_dict(best_state)
teacher.eval()


In [ ]:
# --- Helpers hierarchy for pseudo-labeling ---

def build_child2parents(tree):
    """
    tree : {parent: [children]}
    return : {child: [parents]}
    """
    child2parents = {}
    for parent, children in tree.items():
        for c in children:
            child2parents.setdefault(c, []).append(parent)
    return child2parents

child2parents = build_child2parents(class2hierarchy)


def expand_with_parents(label, child2parents, include_self=True, max_parents=2):
    """
    Pour les pseudo-labels : classe enfant + quelques parents directs.
    """
    expanded = []
    if include_self:
        expanded.append(label)

    parents = child2parents.get(label, [])
    expanded.extend(parents[:max_parents])
    return sorted(set(expanded))


def consistency_loss(student_logits, teacher_logits):
    """
    Consistency sur logits normalisés : plus stable pour inner-product.
    """
    s = F.normalize(student_logits, dim=1)
    t = F.normalize(teacher_logits, dim=1)
    return F.mse_loss(s, t)


def sigmoid_rampup(current, rampup_length):
    if rampup_length == 0:
        return 1.0
    current = max(0.0, min(current, rampup_length))
    phase = 1.0 - current / rampup_length
    return float(np.exp(-5.0 * phase * phase))


def generate_pseudo_labels(
    teacher,
    unlabeled_pids,
    batch_size,
    base_threshold,
    child2parents,
    pid2idx,
    X_train,
    n_classes,
    threshold_min=0.20,
    threshold_decay=None,
    max_new_per_round=1000,
    margin_min=0.03,
):
    """
    Génère des pseudo-labels conservateurs à partir du teacher.
    On garde seulement les cas très confiants.
    """
    teacher.eval()

    dataset_u = UnlabeledEmbeddingDataset(unlabeled_pids)
    loader_u = DataLoader(dataset_u, batch_size=batch_size, shuffle=False)

    new_labels = {}
    confident_pids = []

    with torch.no_grad():
        for batch in loader_u:
            X = batch["X"].to(device)
            pids = batch["pid"]

            logits = teacher(X, use_dropout=False)
            probs = torch.sigmoid(logits).cpu().numpy()

            for pid, p in zip(pids, probs):
                # top-1 et top-2
                sorted_idx = np.argsort(p)[::-1]
                top1 = int(sorted_idx[0])
                top2 = int(sorted_idx[1])
                p1 = float(p[top1])
                p2 = float(p[top2])

                if p1 < base_threshold:
                    continue

                # marge dynamique : p1 doit être clairement devant p2
                dyn_margin = margin_min + 0.5 * max(0.0, 0.5 - p1)
                if p1 < p2 + dyn_margin:
                    continue

                labels = expand_with_parents(top1, child2parents, include_self=True)
                new_labels[int(pid)] = labels
                confident_pids.append(int(pid))

                if len(new_labels) >= max_new_per_round:
                    break
            if len(new_labels) >= max_new_per_round:
                break

    remaining_unlabeled = [pid for pid in unlabeled_pids if pid not in new_labels]

    new_threshold = base_threshold
    if threshold_decay is not None and len(confident_pids) > 0:
        new_threshold = max(threshold_min, base_threshold * threshold_decay)

    print(f"[Pseudo-labeling] New pseudo-labeled samples: {len(new_labels)} "
          f"(remaining unlabeled: {len(remaining_unlabeled)})")
    print(f"[Pseudo-labeling] Threshold used: {base_threshold:.4f} -> next: {new_threshold:.4f}")

    return new_labels, remaining_unlabeled, new_threshold


# --- Regularization + EMA-based self-training ---

# on repart du modèle + teacher sortis du warmup
model.dropout.p = 0.4          # au cas où
teacher.eval()

epochs = 80
patience = 25
wait = 0
pseudo_update_freq = 5         # pseudo-labeling toutes les N epochs
lambda_max = 0.2               # poids max de la consistency
rampup_length = 20             # ramp-up sur les 20 premières epochs ST

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=2e-3)
alpha_ema = 0.995

# seuil initial : Q3 sur les silvers (déjà calculé)
threshold = float(Q2) + 0.02
threshold_decay = 0.99

best_teacher_state = copy.deepcopy(teacher.state_dict())
best_val_f1 = 0.0

# current_labels = silver + pseudo au fur et à mesure
current_labels = dict(silver_dataset)
initial_silver_pids = set(silver_dataset.keys())

# pool unlabeled
unlabeled_set = set(unlabeled_pids)

print("\n===== SELF-TRAINING START =====\n")

for epoch in range(1, epochs + 1):

    # (re)construction du loader pour (silver + pseudo)
    train_pids_full = list(current_labels.keys())
    train_dataset_full = MultiLabelDataset(train_pids_full, current_labels)
    train_loader_full = DataLoader(train_dataset_full, batch_size=64, shuffle=True)

    model.train()
    total_loss = 0.0

    # lambda de consistency avec ramp-up
    lambda_cons = lambda_max * sigmoid_rampup(epoch, rampup_length)

    # ratio de pseudo-labels (pour info/logs)
    n_pseudo = len(current_labels) - len(initial_silver_pids)
    pseudo_ratio = n_pseudo / max(1, len(current_labels))
    print(f"[ST Epoch {epoch}] pseudo_ratio={pseudo_ratio:.3f}, lambda_cons={lambda_cons:.3f}")

    for batch in tqdm(train_loader_full, desc=f"ST Epoch {epoch}/{epochs}"):
        X = batch["X"].to(device)
        y = batch["y"].to(device)

        # label smoothing léger
        y_smooth = y * 0.9 + 0.05

        # --- student ---
        student_logits = model(X, use_dropout=True)
        sup_loss = F.binary_cross_entropy_with_logits(student_logits, y_smooth)

        # --- teacher ---
        with torch.no_grad():
            teacher_logits = teacher(X, use_dropout=False)

        cons_loss = consistency_loss(student_logits, teacher_logits)

        loss = sup_loss + lambda_cons * cons_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader_full)
    print(f"[ST Epoch {epoch}] train_loss={avg_loss:.4f}")

    # --- EMA update du teacher ---
    ema_update(teacher, model, alpha_ema)
    teacher.eval()

    # --- Validation ---
    model.eval()
    f1s, f1m, p1, p3 = evaluate(model, val_loader)
    val_p1_list2.append(p1)
    val_p3_list2.append(p3)
    val_f1_list2.append(f1s)
    print(f"[ST Epoch {epoch}] val F1={f1s:.4f}, P@1={p1:.4f}, P@3={p3:.4f}")

    if f1s > best_val_f1:
        best_val_f1 = f1s
        best_teacher_state = copy.deepcopy(teacher.state_dict())
        wait = 0
        print(f"  -> New best teacher (F1={best_val_f1:.4f})")
    else:
        wait += 1
        print(f"  No improvement for {wait} epoch(s).")

    if wait >= patience:
        print(f"\nEarly stopping in self-training after epoch {epoch}.")
        break

    # --- Pseudo-labeling périodique ---
    if (epoch % pseudo_update_freq == 0) and len(unlabeled_set) > 0:
        unlabeled_pids_curr = sorted(list(unlabeled_set))

        new_pseudo, remaining_unlabeled, new_thr = generate_pseudo_labels(
            teacher=teacher,
            unlabeled_pids=unlabeled_pids_curr,
            batch_size=64,
            base_threshold=threshold,
            child2parents=child2parents,
            pid2idx=pid2idx,
            X_train=X_train,
            n_classes=n_classes,
            threshold_min=0.20,
            threshold_decay=threshold_decay,
            max_new_per_round=1000,   # limite de sécurité
        )

        # ajout des nouveaux pseudo-labels
        for pid, labels in new_pseudo.items():
            current_labels[pid] = labels

        unlabeled_set = set(remaining_unlabeled)
        threshold = new_thr

        print(f"[ST Epoch {epoch}] After pseudo-labeling: "
              f"total labeled={len(current_labels)}, remaining unlabeled={len(unlabeled_set)}")

print("\n===== SELF-TRAINING END =====")
teacher.load_state_dict(best_teacher_state)
print(f"Best self-training F1 (teacher) = {best_val_f1:.4f}")


In [ ]:
import matplotlib.pyplot as plt

def plot_all_metrics(results_dict):
    """
    Similar to plot from utils.py
    Plot F1, P@1, and P@3 curves for 2 models.

    results_dict must be of form:
    {
        "F1": {"Model 1": [...], "Model 2": [...]},
        "P@1": {"Model 1": [...], "Model 2": [...]},
        "P@3": {"Model 1": [...], "Model 2": [...]},
    }
    """

    metrics = ["F1", "P@1", "P@3"]
    plt.figure(figsize=(16, 4))

    for i, metric in enumerate(metrics):
        plt.subplot(1, 3, i+1)
        for model_name, values in results_dict[metric].items():
            plt.plot(values, label=model_name)

        plt.title(metric + " over epochs")
        plt.xlabel("Epoch")
        plt.ylabel(metric)
        plt.grid(True)
        plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
results = {
    "F1": {
        "Model 1": val_f1_list,
        "Model 2": val_f1_list2,
    },
    "P@1": {
        "Model 1": val_p1_list,
        "Model 2": val_p1_list2,
    },
    "P@3": {
        "Model 1": val_p3_list,
        "Model 2": val_p3_list2,
    }
}

# Graphics
print("\nGraph")
plot_all_metrics(results)

print(val_f1_list)

In [ ]:
# Submission step

# loading test_ids
test_ids = []
with open(TEST_CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        pid, _ = line.strip().split("\t", 1)
        test_ids.append(int(pid))
print(len(test_ids))

# Model selection : top-2 / top-3
def select_k(prob, min_k=2, max_k=3):
    idx = np.argsort(prob)[::-1]     # top probs first
    top3 = idx[:max_k]
    # if the 3rd is too far from the 2nd -> keep only 2 labels
    if prob[top3[2]] < 0.33 * prob[top3[1]]:
        return top3[:2]
    return top3


# Regularized model
model.eval()
X_test = X_test.to(device)

# prediction
preds = []
batch_size = 64

with torch.no_grad():
    for start in tqdm(range(0, len(X_test), batch_size)):
        batch = X_test[start:start+batch_size]
        logits = model(batch)
        probs = torch.sigmoid(logits).cpu().numpy()

        for p in probs:
            labels = select_k(p)                     
            preds.append([str(x) for x in labels]) 

# Save
OUT_DIR = Path("Submission")
OUT_DIR.mkdir(exist_ok=True)
OUT_PATH = OUT_DIR / "submission_self_reg.csv"

with open(OUT_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "label"])
    for pid, labels in zip(test_ids, preds):
        w.writerow([pid, ",".join(labels)])

print(f"Submission saved -> {OUT_PATH}")

# Not Reg
save_model.eval()
X_test = X_test.to(device)

# prediction
preds = []
batch_size = 64

with torch.no_grad():
    for start in tqdm(range(0, len(X_test), batch_size)):
        batch = X_test[start:start+batch_size]
        logits = save_model(batch)
        probs = torch.sigmoid(logits).cpu().numpy()

        for p in probs:
            labels = select_k(p)                     
            preds.append([str(x) for x in labels]) 

# Save
OUT_DIR = Path("Submission")
OUT_DIR.mkdir(exist_ok=True)
OUT_PATH = OUT_DIR / "submission_self_notreg.csv"

with open(OUT_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "label"])
    for pid, labels in zip(test_ids, preds):
        w.writerow([pid, ",".join(labels)])

print(f"Submission saved -> {OUT_PATH}")


In [ ]:
# Comparison with real results

def load_csv_labels(path):
    """
    Load a CSV file formatted as:
    id,label1 label2 label3
    Returns {id: [labels]}
    """
    out = {}
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            pid = int(row["id"])
            labels = [int(x) for x in row["label"].replace(",", " ").split()]
            out[pid] = labels
    return out


def score_submission_csv(pred_path, gold_path):
    """
    Score a submission CSV using your GOLD CSV.
    Computes: F1 (samples), P@1, P@3
    """
    pred = load_csv_labels(pred_path)
    gold = load_csv_labels(gold_path)

    # Sorted IDs intersection
    ids = sorted(set(pred.keys()) & set(gold.keys()))

    num_classes = 531

    Y_true = []
    Y_pred = []
    Y_scores = []

    for pid in ids:
        true_vec  = np.zeros(num_classes)
        pred_vec  = np.zeros(num_classes)
        score_vec = np.zeros(num_classes)

        true_vec[gold[pid]] = 1
        pred_vec[pred[pid]] = 1
        score_vec[pred[pid]] = 1.0

        Y_true.append(true_vec)
        Y_pred.append(pred_vec)
        Y_scores.append(score_vec)

    f1s = f1_score(Y_true, Y_pred, average="samples")
    P1 = precision_at_1(Y_true, Y_scores)
    P3 = precision_at_3(Y_true, Y_scores)

    print(f"Sample F1 : {f1s:.4f}")
    print(f"P@1 : {P1:.2f}")
    print(f"P@3 : {P3:.2f}")

score_submission_csv(
    pred_path="Submission/submission_self_reg.csv",
    gold_path="Gold/gold.csv"
)